In [1]:
import pandas as pd

columns = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent',
    'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count',
    'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

DATA_PATH = "s3://cmpe281-risk-detection/cmpe_281_data_models/data/raw/KDDTrain+.csv"
df = pd.read_csv(DATA_PATH, header=None, names=columns)

# 2. Drop metadata
df.drop(columns=['difficulty_level'], inplace=True)

# 3. Binary target
df['target'] = df['label'].apply(lambda x: 0 if x == 'normal' else 1)
df.drop(columns=['label'], inplace=True)

# 4. Encode categoricals
df = pd.get_dummies(df, columns=['protocol_type', 'service', 'flag'])

# 5. Separate features and target
X = df.drop(columns=['target'])
y = df['target']

print("Shape:", df.shape)
print("Object cols left:", list(X.select_dtypes(include='object').columns))
print("Target balance:\n", y.value_counts())


KeyboardInterrupt



# Train Random Forest

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
# step 1: Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
# step 2 Create random forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [ ]:
# step 3: Train and predict on test data

rf_model.fit(X_train, y_train)

# predict
y_pred_rf = rf_model.predict(X_test)

In [ ]:
# results

print("Test Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

In [ ]:
# check training accuracy
y_train_pred_rf = rf_model.predict(X_train)
print("Train Accuracy:", accuracy_score(y_train, y_train_pred_rf))